# 36 -- Section 01e Step 6: training (CD-1 update)

Computes the first CD-1 update for the Rumble in the Bronx x h1 worked
example, dW+ = lr*(v0+ q0+ - vhat+ q1+) and dW- = lr*(v0- q0- - vhat- q1-),
for both the 01c (b=0) and 01e (b=-1) scenarios, using exactly notebook 31's
formula. Reuses notebook 31/32/35's exact seeds, initial weights and sampled
hidden state. Does not touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import torch

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "step6_training" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step6_training.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  35  (35_step5_visible_reconstruction.ipynb)
this notebook is:           36  (36_step6_training.ipynb)


## Part 1 -- Reuse notebook 31/32/35's exact setup

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
results35_path = out_dir / "35_step5_visible_reconstruction_results.json"
for p in (train_path, test_path, results31_path, results35_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474

results31 = json.loads(results31_path.read_text())
results35 = json.loads(results35_path.read_text())
REAL_SEED = results31["seeds"]["plus_seed_reused"]
MINUS_SEED = results31["seeds"]["minus_seed_trained"]
N_HIDDEN = int(results31["architecture_match"]["128 hidden units"].split("=")[1])
LR = results31["worked_example_plus"]["lr"]
H1_IDX = 0
print(f"plus_seed={REAL_SEED}  minus_seed={MINUS_SEED}  N_HIDDEN={N_HIDDEN}  LR={LR}")

all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_movies = len(all_movies)
movie_to_col = {m: j for j, m in enumerate(all_movies)}

movies = pd.read_csv(data_dir / "movie.csv")
RUMBLE_ID = int(movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)].iloc[0]["movieId"])
RUMBLE_TITLE = movies[movies["movieId"] == RUMBLE_ID].iloc[0]["title"]
rumble_col = movie_to_col[RUMBLE_ID]
MEDIAN_USER = results31["worked_example_plus"]["user"]

user_rows = train_hyp[train_hyp["userId"] == MEDIAN_USER]
v0_full = torch.zeros(n_movies)
mask0_full = torch.zeros(n_movies)
for row in user_rows.itertuples():
    v0_full[movie_to_col[row.movieId]] = row.a
    mask0_full[movie_to_col[row.movieId]] = 1.0
a_val = v0_full[rumble_col].item()
mask0b = mask0_full.unsqueeze(0)

g_init_p = torch.Generator().manual_seed(REAL_SEED)
W_plus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_p) * 0.01
hbias_plus_init = torch.zeros(N_HIDDEN)
vbias_plus_init = torch.zeros(n_movies)

g_init_m = torch.Generator().manual_seed(MINUS_SEED)
W_minus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_m) * 0.01
hbias_minus_init = torch.zeros(N_HIDDEN)
vbias_minus_init = torch.zeros(n_movies)

w_plus_rumble = W_plus_init[rumble_col, H1_IDX].item()
w_minus_rumble = W_minus_init[rumble_col, H1_IDX].item()
print(f"user={MEDIAN_USER}  a[Rumble]={a_val:.6f}")
print(f"w+[Rumble,h1]={w_plus_rumble:.6f}  w-[Rumble,h1]={w_minus_rumble:.6f}")

plus_seed=44  minus_seed=45  N_HIDDEN=128  LR=0.05
user=24127  a[Rumble]=0.777778
w+[Rumble,h1]=0.006433  w-[Rumble,h1]=-0.009362


## Part 2 -- Full CD-1 round trip for both channels, both scenarios

v0 -> h0 (sample s, same seed as notebook 31) -> v1 (masked reconstruction) -> h1.
Only the Rumble entry of v0 differs between 01c and 01e.

In [3]:
b_val = -1.0
v0_plus_01c = v0_full.clone()
v0_minus_01c = v0_full.clone()
v0_plus_01e = v0_full.clone(); v0_plus_01e[rumble_col] = a_val + b_val
v0_minus_01e = v0_full.clone(); v0_minus_01e[rumble_col] = a_val - b_val

def cd1_round_trip(v0_vec, W, hbias, vbias, mask, seed):
    v0b = v0_vec.unsqueeze(0)
    h0_prob = torch.sigmoid(v0b @ W + hbias)
    g = torch.Generator().manual_seed(seed)
    h0_sample = torch.bernoulli(h0_prob, generator=g)
    v1_prob = torch.sigmoid(h0_sample @ W.t() + vbias) * mask
    h1_prob = torch.sigmoid(v1_prob @ W + hbias)
    return h0_prob, h0_sample, v1_prob, h1_prob

h0_plus_01c, s_plus_01c, v1_plus_01c, h1_plus_01c = cd1_round_trip(v0_plus_01c, W_plus_init, hbias_plus_init, vbias_plus_init, mask0b, REAL_SEED)
h0_minus_01c, s_minus_01c, v1_minus_01c, h1_minus_01c = cd1_round_trip(v0_minus_01c, W_minus_init, hbias_minus_init, vbias_minus_init, mask0b, MINUS_SEED)
h0_plus_01e, s_plus_01e, v1_plus_01e, h1_plus_01e = cd1_round_trip(v0_plus_01e, W_plus_init, hbias_plus_init, vbias_plus_init, mask0b, REAL_SEED)
h0_minus_01e, s_minus_01e, v1_minus_01e, h1_minus_01e = cd1_round_trip(v0_minus_01e, W_minus_init, hbias_minus_init, vbias_minus_init, mask0b, MINUS_SEED)

print("sampled hidden state identical 01c vs 01e:",
      "plus:", torch.equal(s_plus_01c, s_plus_01e), " minus:", torch.equal(s_minus_01c, s_minus_01e))
print("full v1 (masked reconstruction) identical 01c vs 01e:",
      "plus:", torch.equal(v1_plus_01c, v1_plus_01e), " minus:", torch.equal(v1_minus_01c, v1_minus_01e))

q0_plus = {"01c": h0_plus_01c[0, H1_IDX].item(), "01e": h0_plus_01e[0, H1_IDX].item()}
q0_minus = {"01c": h0_minus_01c[0, H1_IDX].item(), "01e": h0_minus_01e[0, H1_IDX].item()}
vhat_plus = {"01c": v1_plus_01c[0, rumble_col].item(), "01e": v1_plus_01e[0, rumble_col].item()}
vhat_minus = {"01c": v1_minus_01c[0, rumble_col].item(), "01e": v1_minus_01e[0, rumble_col].item()}
q1_plus = {"01c": h1_plus_01c[0, H1_IDX].item(), "01e": h1_plus_01e[0, H1_IDX].item()}
q1_minus = {"01c": h1_minus_01c[0, H1_IDX].item(), "01e": h1_minus_01e[0, H1_IDX].item()}

assert abs(vhat_plus["01c"] - results35["vhat_plus"]["01c"]) < 1e-6
assert abs(vhat_minus["01c"] - results35["vhat_minus"]["01c"]) < 1e-6
assert abs(q1_plus["01c"] - 0.4601) < 5e-4 and abs(q1_minus["01c"] - 0.4652) < 5e-4
print("q1 and vhat for the 01c scenario match notebook 31/35 and the page's Step 8 table. OK.")

v0_rumble_plus = {"01c": v0_plus_01c[rumble_col].item(), "01e": v0_plus_01e[rumble_col].item()}
v0_rumble_minus = {"01c": v0_minus_01c[rumble_col].item(), "01e": v0_minus_01e[rumble_col].item()}
print(json.dumps({"q0_plus": q0_plus, "q0_minus": q0_minus, "vhat_plus": vhat_plus, "vhat_minus": vhat_minus, "q1_plus": q1_plus, "q1_minus": q1_minus}, indent=2))

sampled hidden state identical 01c vs 01e: plus: True  minus: True
full v1 (masked reconstruction) identical 01c vs 01e: plus: True  minus: True
q1 and vhat for the 01c scenario match notebook 31/35 and the page's Step 8 table. OK.
{
  "q0_plus": {
    "01c": 0.44741639494895935,
    "01e": 0.4458264112472534
  },
  "q0_minus": {
    "01c": 0.44883793592453003,
    "01e": 0.44652310013771057
  },
  "vhat_plus": {
    "01c": 0.48846325278282166,
    "01e": 0.48846325278282166
  },
  "vhat_minus": {
    "01c": 0.4632098078727722,
    "01e": 0.4632098078727722
  },
  "q1_plus": {
    "01c": 0.46008285880088806,
    "01e": 0.46008285880088806
  },
  "q1_minus": {
    "01c": 0.4651552140712738,
    "01e": 0.4651552140712738
  }
}


## Part 3 -- CD-1 update: dW+ = lr*(v0+ q0+ - vhat+ q1+), dW- likewise

In [4]:
dW_plus = {}
dW_minus = {}
dB = {}
for scenario in ("01c", "01e"):
    dW_plus[scenario] = LR * (v0_rumble_plus[scenario] * q0_plus[scenario] - vhat_plus[scenario] * q1_plus[scenario])
    dW_minus[scenario] = LR * (v0_rumble_minus[scenario] * q0_minus[scenario] - vhat_minus[scenario] * q1_minus[scenario])
    dB[scenario] = (dW_plus[scenario] - dW_minus[scenario]) / 2

print(f"dW+ : 01c={dW_plus['01c']:.6f}  01e={dW_plus['01e']:.6f}")
print(f"dW- : 01c={dW_minus['01c']:.6f}  01e={dW_minus['01e']:.6f}")
print(f"dB  : 01c={dB['01c']:.6f}  01e={dB['01e']:.6f}")

assert abs(dW_plus["01c"] - 0.0062) < 5e-4, "01c plus update must match the page's Step 8 table"
assert abs(dW_minus["01c"] - 0.0067) < 5e-4, "01c minus update must match the page's Step 8 table"
print("01c column matches the values already shown in 01c Step 8. OK.")

dW+ : 01c=0.006163  01e=-0.016190
dW- : 01c=0.006682  01e=0.028918
dB  : 01c=-0.000259  01e=-0.022554
01c column matches the values already shown in 01c Step 8. OK.


## Part 4 -- Save for Step 6 of section 01e

In [5]:
results = {
    "notebook": this_name,
    "reused_from": "31_step8_minus_channel.ipynb, 32_step2_hidden_input.ipynb, 35_step5_visible_reconstruction.ipynb",
    "movie": RUMBLE_TITLE,
    "user": MEDIAN_USER,
    "hidden_unit": "h1 (index 0)",
    "lr": LR,
    "v0_plus": v0_rumble_plus,
    "v0_minus": v0_rumble_minus,
    "q0_plus": q0_plus,
    "q0_minus": q0_minus,
    "vhat_plus": vhat_plus,
    "vhat_minus": vhat_minus,
    "q1_plus": q1_plus,
    "q1_minus": q1_minus,
    "dW_plus": dW_plus,
    "dW_minus": dW_minus,
    "dB": dB,
}
results_path = out_dir / "36_step6_training_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps(results, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/36_step6_training_results.json
{
  "notebook": "36_step6_training.ipynb",
  "reused_from": "31_step8_minus_channel.ipynb, 32_step2_hidden_input.ipynb, 35_step5_visible_reconstruction.ipynb",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "user": 24127,
  "hidden_unit": "h1 (index 0)",
  "lr": 0.05,
  "v0_plus": {
    "01c": 0.7777777910232544,
    "01e": -0.2222222089767456
  },
  "v0_minus": {
    "01c": 0.7777777910232544,
    "01e": 1.7777777910232544
  },
  "q0_plus": {
    "01c": 0.44741639494895935,
    "01e": 0.4458264112472534
  },
  "q0_minus": {
    "01c": 0.44883793592453003,
    "01e": 0.44652310013771057
  },
  "vhat_plus": {
    "01c": 0.48846325278282166,
    "01e": 0.48846325278282166
  },
  "vhat_minus": {
    "01c": 0.4632098078727722,
    "01e": 0.4632098078727722
  },
  "q1_plus": {
    "01c": 0.46008285880088806,
    "01e": 0.46008285880088806
  },
  "q1_minus": {
    "01c": 0.465155